In [5]:
import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath("."))

from data_pipeline.score_reddit import run_scoring
from data_pipeline.filtering import ThreadFilter

from simulations.naive_debate import run_naive_simulation
from simulations.reddit_aligned import run_reddit_simulation
from simulations.moderated_reddit import run_moderated_simulation

from evaluations.evaluate_debates import run_batch_evaluation
from evaluations.summarize import summarize_evaluations

RAW_DATA_PATH = Path("data/raw_submissions.jsonl")
SCORED_DATA_PATH = Path("data/scored_submissions.jsonl")
SIM_SEEDS_PATH = Path("data/sim_seeds.jsonl")
SIM_OUT_DIR = Path("sim_debate_records")
EVAL_OUT_DIR = Path("eval_records")

SIM_OUT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Environment setup complete. Ready to run pipeline.")

Environment setup complete. Ready to run pipeline.


In [7]:
if RAW_DATA_PATH.exists():
    print(f"Raw data already exists at {RAW_DATA_PATH}. Skipping scraper.")
else:
    print("Running Reddit scraper...")
    !uv run python data_pipeline/scraper.py

    if Path("data.jsonl").exists():
        Path("data.jsonl").rename(RAW_DATA_PATH)
        print(f"Moved raw scraped data to {RAW_DATA_PATH}")
    else:
        print("Warning: Could not find scraper output.")

Raw data already exists at data\raw_submissions.jsonl. Skipping scraper.


In [ ]:
print("Starting DeBERTa toxicity scoring (this may take a moment)...")
run_scoring(
    input_path=RAW_DATA_PATH, 
    output_path=SCORED_DATA_PATH
)

Starting DeBERTa toxicity scoring (this may take a moment)...
Scored thread 1: 1s9zdxj
Scored thread 2: 1s9vtjx
Scored thread 3: 1s9tw3l
Scored thread 4: 1s9t94r
Scored thread 5: 1s9ju98
Scored thread 6: 1s9ju8z
Scored thread 7: 1s9gsyd
Scored thread 8: 1s8ry7t
Scored thread 9: 1s8in4f
Scored thread 10: 1s7qxe9
Scored thread 11: 1s7n09k
Scored thread 12: 1s66a6g
Scored thread 13: 1s3uc44
Scored thread 14: 1s3o4o3
Scored thread 15: 1s3h8xj
Scored thread 16: 1s2qaun
Scored thread 17: 1s2m1gt
Scored thread 18: 1s274br
Scored thread 19: 1s1qnq6
Scored thread 20: 1s1p5u1
Scored thread 21: 1s1n0a3
Scored thread 22: 1rzz92k
Scored thread 23: 1rzdili
Scored thread 24: 1rzc7n4
Scored thread 25: 1rz058n
Scored thread 26: 1ryx224
Scored thread 27: 1ryx21v
Scored thread 28: 1ryr1ww
Scored thread 29: 1ry96ub
Scored thread 30: 1ry689u
Scored thread 31: 1rxvimc
Scored thread 32: 1rx6yqn
Scored thread 33: 1rwq0ar
Scored thread 34: 1rwks6t
Scored thread 35: 1rvctnu
Processing r/technology...
  Found 1 

In [ ]:
print("Filtering for toxic debate chains...")

filter_engine = ThreadFilter(
    max_threads=5,         
    chain_threshold=0.75,  
    chain_length=2         
)

selected_threads = filter_engine.run_filtering(
    input_path=SCORED_DATA_PATH, 
    output_path=SIM_SEEDS_PATH
)

In [3]:
ROUNDS = 3
MODEL = "gemma4:e4b"
JUDGE_MODEL = "llama3.2:3b"

In [ ]:
print(f"--- 1. Running Naive Simulation ---")
run_naive_simulation(
    model=MODEL, 
    rounds=ROUNDS, 
    out_dir=SIM_OUT_DIR
)

print(f"\n--- 2. Running Reddit-Aligned Simulation ---")
run_reddit_simulation(
    input_path=SIM_SEEDS_PATH,
    model=MODEL,
    rounds=ROUNDS,
    out_dir=SIM_OUT_DIR,
    submission_index=0
)

print(f"\n--- 3. Running Moderated Reddit-Aligned Simulation ---")
run_moderated_simulation(
    input_path=SIM_SEEDS_PATH,
    model=MODEL,
    judge_model=JUDGE_MODEL,
    rounds=ROUNDS,
    out_dir=SIM_OUT_DIR,
    toxicity_threshold=0.00042,
    submission_index=0
)

print("\nAll simulations complete!")

--- 1. Running Naive Simulation ---
Starting Naive Debate on 'abortion rights' using gemma4:e4b
Round 1 (ID: 3e146)
--------------------------------------------------
Pro-Choice Advocate: My position is unequivocally that abortion access must be a legal right, rooted in fundamental bodily autonomy and privacy. This debate must center on the rights of the individual, not on hypothetical moral tests.

Every person has the right to govern their own physical life and medical decisions. Restricting abortion severely impacts public health and women's futures. We must trust the medical judgment of the providers and uphold the constitutional right to privacy, ensuring that robust and safe care remains accessible to all.
--------------------------------------------------
Pro-Life Advocate: Bodily autonomy is not a shield that allows one person to unilaterally end another human life. The law cannot, and should not, grant the right to dispose of a distinct, developing human being. A person's righ

In [4]:
print("Judging the generated simulations...")
run_batch_evaluation(
    sim_dir=SIM_OUT_DIR, 
    out_dir=EVAL_OUT_DIR, 
    judge_model=JUDGE_MODEL
)

print("\nAggregating final metrics...")
summarize_evaluations(eval_dir=EVAL_OUT_DIR)

Judging the generated simulations...
Judging moderated_reddit_abortion_rights_1s3uc44_0cbf6_gemma4_e4b.jsonl using llama3.2:3b...
Evaluation saved to eval_records\eval_moderated_reddit_abortion_rights_1s3uc44_0cbf6_gemma4_e4b.jsonl
Judging moderated_reddit_abortion_rights_1s3uc44_7182b_llama3.1_8b.jsonl using llama3.2:3b...
Evaluation saved to eval_records\eval_moderated_reddit_abortion_rights_1s3uc44_7182b_llama3.1_8b.jsonl
Judging naive_abortion_rights_3e146_gemma4_e4b.jsonl using llama3.2:3b...
Evaluation saved to eval_records\eval_naive_abortion_rights_3e146_gemma4_e4b.jsonl
Judging naive_abortion_rights_647f4_llama3.1_8b.jsonl using llama3.2:3b...
Evaluation saved to eval_records\eval_naive_abortion_rights_647f4_llama3.1_8b.jsonl
Judging reddit_abortion_rights_1s3uc44_b05d4_llama3.1_8b.jsonl using llama3.2:3b...
Evaluation saved to eval_records\eval_reddit_abortion_rights_1s3uc44_b05d4_llama3.1_8b.jsonl
Judging reddit_abortion_rights_1s3uc44_f0b36_gemma4_e4b.jsonl using llama3.2:3